In [5]:
# Install Dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes trl

Failed to pip install flash-attn, won't use it now.

In [7]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU name: NVIDIA A100-SXM4-40GB
GPU memory: 42.4 GB


FULL FINE-TUNING:
- Updates ALL model parameters (billions of weights)
- Requires massive GPU memory
- Risk of catastrophic forgetting

LoRA FINE-TUNING :
- Freezes the original model weights
- Adds small "adapter" matrices to specific layers
- Only trains these adapters (~0.1-1% of total parameters)
- Original knowledge preserved, new capability added

In a standard Transformer layer, the input $X$ (your legal text) is multiplied by the weight matrix $W$.
$$Output = W \cdot X$$

With LoRA, we add a parallel mathematical "detour":
$$Output = (W \cdot X) + (B \cdot A \cdot X)$$

*   **The Frozen Path ($W \cdot X$):** This is the "Base Intelligence." It keeps the model's ability to understand English, grammar, and general logic. This prevents **Catastrophic Forgetting**—the model won't suddenly forget how to speak English just because you taught it Law.
*   **The LoRA Path ($B \cdot A \cdot X$):** This is the "Legal Specialist." It acts as a **Correction Factor**.

**Example in Legal AI:**
Suppose the input $X$ is the word "Sanction."
1.  The **Frozen Path ($W$)** predicts the most common meaning: *"An official permission or approval."*
2.  The **LoRA Path ($B \cdot A$)** (which was trained on legal penalty documents) produces a mathematical "steer" or "offset."
3.  When you **add** them together, the offset shifts the probability distribution so that "Penalty/Punishment" becomes the top choice.


In [22]:
# Load a Pre-trained Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Using Qwen2.5-1.5B, small but capable
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantisation config
# This compresses the model to use ~4x less memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load in 4-bit precision
    bnb_4bit_quant_type="nf4",             # NormalFloat4 quantisation type
    bnb_4bit_compute_dtype=torch.bfloat16, # update Compute Dtype to BFloat16, supported by A100
    bnb_4bit_use_double_quant=True,        # Double quantisation for extra savings
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,             # <--- CHANGED: Match the compute dtype
#     attn_implementation="flash_attention_2" # <--- NEW: A100 exclusive! Massive speed/memory boost
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Check model size
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")
print(f"Trainable (before LoRA): {trainable:,}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Total parameters: 888,616,448
Trainable (before LoRA): 233,518,592


In [ ]:
# Test the Base Model BEFORE Fine-Tuning
def ask_model(prompt, model, tokenizer, max_new_tokens=200):
    """Helper function to query the model"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Test with legal questions
test_questions = [
    "A woman in Malawi is experiencing domestic violence. What legal steps can she take to protect herself? Respond in a structured, empathetic way.",
    "Explain what a protection order is and how to apply for one. Use simple language suitable for someone with no legal background.",
    "A journalist has been detained for reporting on government corruption. What are their rights under international law?",
]

print("=" * 60)
print("BASE MODEL RESPONSES (BEFORE FINE-TUNING)")
print("=" * 60)
for i, q in enumerate(test_questions):
    print(f"\n--- Question {i+1} ---")
    print(f"Q: {q[:80]}...")
    response = ask_model(q, model, tokenizer)
    print(f"A: {response[:500]}")
    print()

BASE MODEL RESPONSES (BEFORE FINE-TUNING)

--- Question 1 ---
Q: A woman in Malawi is experiencing domestic violence. What legal steps can she ta...
A: If a woman in Malawi experiences domestic violence, she has several legal options available to her for protection and safety:

1. **Report the Incident**: The first step is to report the incident to local authorities such as the police or relevant government agencies. In Malawi, reporting abuse may be done through the national hotline number 987 which provides assistance in case of emergency.

2. **Seek Legal Protection Orders**: Malawi recognizes domestic violence as a crime and offers legal pr


--- Question 2 ---
Q: Explain what a protection order is and how to apply for one. Use simple language...
A: A protection order is like a special kind of agreement that helps keep people safe from harm or abuse. It's usually made when someone feels in danger because their partner or family member is being mean or hurtful.

To get a protection 

## Analysis of the base model output
### Q1:
'national hotline number 987' is a **hallucination**. The actual Gender-Based Violence (GBV) crisis line in Malawi is 5600, and the police emergency number is 997.  
> A base model cannot be trusted to memorize specific, localized facts and phone numbers. And it need fine-tuning to provide specific format for legal aid.

### Q2:
'talk to a grown-up' is a misinterpreted of the original prompt: "simple" as "infantile" and explained it as if talking to a 5-year-old child.
> A base model lack proper Alignment for legal empathy. It needs to learn the "Style and Format" to speak in respectful adult language.

### Q3:
'Article 19 of the International Covenant on Civil and Political Rights (ICCPR)', this time no hallucination, but the base model stops at a very generic, Wikipedia-level summary. Need further detail on actionable legal mechanisms.
> A base model lacks the Domain Expertise in Civil Rights.

In [14]:
# Prepare Training Data

# Creating a small synthetic dataset that teaches the model
# HOW to respond to legal aid questions — the STYLE and STRUCTURE,
# not legal facts (which should come from RAG at inference time)

# This demonstrates the key insight: fine-tuning for FORM, not FACTS

training_data = [
    {
        "instruction": "A woman is being abused by her partner and wants to know her options. Respond with empathy and provide structured guidance.",
        "response": """I'm sorry to hear about this situation. Your safety is the most important thing, and there are legal protections available to you.

**Immediate Steps:**
1. If you are in immediate danger, contact emergency services right away.
2. Reach out to a local women's shelter or domestic violence hotline for safe accommodation.

**Legal Options:**
A protection order (also called a restraining order) is a court order that legally requires the abuser to stay away from you. To apply for one:
- Contact your nearest court or legal aid office
- You can describe what happened in your own words — you don't need a lawyer to start the process
- A judge will review your application, usually within 24-48 hours for urgent cases

**Free Legal Help:**
Many organisations offer free legal representation for domestic violence cases. A pro bono lawyer can help you through the entire process at no cost.

Would you like me to help you find specific resources in your area?"""
    },
    {
        "instruction": "Explain what bail is to someone who has never dealt with the legal system before. Use simple, clear language.",
        "response": """Bail is essentially a promise — backed by money or conditions — that allows someone who has been arrested to leave jail while they wait for their court date.

**How it works in simple terms:**
When someone is arrested, they don't always have to stay in jail until their trial. A judge can decide to release them on bail, which means:
- They can go home and continue their normal life while waiting for their court date
- They must follow certain rules (like reporting to police regularly or not leaving the area)
- Sometimes they or their family must pay a sum of money as a guarantee they'll come back for the trial

**Important things to know:**
- Bail is a RIGHT in many legal systems, not a privilege — the judge needs good reasons to deny it
- If someone cannot afford bail, they can ask for it to be reduced or waived
- A lawyer can make a bail application on behalf of the detained person

The key principle is that people are presumed innocent until proven guilty, so they shouldn't be kept in jail unnecessarily before their trial."""
    },
    {
        "instruction": "A journalist has been arrested for their reporting. Their editor wants to know what legal protections exist. Provide a structured overview.",
        "response": """This is a serious matter, and there are important legal protections for journalists under international law.

**International Legal Protections:**
Freedom of expression is protected under Article 19 of the International Covenant on Civil and Political Rights (ICCPR) and Article 10 of the European Convention on Human Rights. This includes the right to seek, receive, and share information.

**Immediate Steps for the Editor:**
1. Document the arrest: Record the date, time, location, and circumstances
2. Contact a press freedom organisation such as the Committee to Protect Journalists (CPJ) — they can provide emergency legal assistance
3. Engage a local lawyer experienced in media law as quickly as possible
4. Notify your country's embassy or consulate if the journalist is detained abroad

**For the Bail Application:**
A lawyer should argue that:
- The journalist was exercising their right to free expression
- They are not a flight risk and will attend all court proceedings
- Continued detention is not necessary or proportionate

**Key Principle:**
Journalism is not a crime. Legitimate reporting in the public interest is protected activity, and detention for it may violate international human rights standards.

Would you like help connecting with press freedom legal support networks?"""
    },
    {
        "instruction": "Someone asks: 'What is a contract and when is it legally binding?' Explain clearly for a non-lawyer.",
        "response": """A contract is simply an agreement between two or more people (or organisations) that the law will enforce.

**The basics:**
Think of it like a promise with legal teeth. If you and I agree that I'll paint your house for £500, and we both genuinely intend this to be a binding deal, that's a contract — even if we never wrote anything down.

**When is a contract legally binding?**
Generally, you need these elements:
- **Offer**: One party proposes something specific ("I'll paint your house for £500")
- **Acceptance**: The other party agrees to those exact terms ("Yes, deal")
- **Consideration**: Both sides give something of value (I give my labour, you give £500)
- **Intention**: Both parties intend it to be a legal agreement (not just a casual favour)
- **Capacity**: Both parties are legally able to make contracts (adults of sound mind)

**Common misconceptions:**
- Contracts don't always need to be written — verbal contracts can be legally binding
- However, some types MUST be in writing (like property sales or employment contracts)
- Signing something you haven't read doesn't mean you can escape it — always read before you sign

**When might a contract NOT be valid?**
- If someone was forced or tricked into signing (duress or fraud)
- If one party is a minor
- If the contract asks for something illegal"""
    },
    {
        "instruction": "Explain the concept of 'fair trial' rights to a human rights worker who needs to monitor court proceedings.",
        "response": """The right to a fair trial is one of the most fundamental human rights, and understanding its components is essential for effective monitoring.

**Core Fair Trial Rights:**

1. **Right to be heard by an independent, impartial tribunal**
   - The judge must have no personal interest in the case
   - The court must be free from outside pressure (political, media, etc.)

2. **Right to legal representation**
   - Everyone has the right to a lawyer, especially in criminal cases
   - If someone cannot afford a lawyer, one should be provided free of charge for serious offences

3. **Presumption of innocence**
   - The accused is innocent until proven guilty
   - The burden of proof lies on the prosecution, not the defendant

4. **Right to adequate time and facilities to prepare a defence**
   - The accused and their lawyer must have access to all evidence against them
   - They must be given sufficient time to prepare

5. **Right to examine witnesses**
   - The defence must be able to question prosecution witnesses
   - They must be able to call their own witnesses under the same conditions

6. **Right to a public hearing**
   - Trials should generally be open to the public and press
   - Exceptions exist for national security or protection of minors

**What to watch for during monitoring:**
- Is the defendant represented by a lawyer?
- Does the judge appear impartial and attentive?
- Is the defence given equal opportunity to present their case?
- Are proceedings conducted in a language the defendant understands?

These standards come from international instruments including the ICCPR (Articles 14-15) and regional human rights conventions."""
    },
    {
        "instruction": "A community worker asks how AI can help people who cannot afford a lawyer. Provide a hopeful but realistic answer.",
        "response": """This is one of the most exciting areas where technology can make a real difference, and there are already working examples around the world.

**How AI is helping right now:**

1. **Legal information chatbots**
   AI-powered assistants can help people understand their rights in plain language, in their own language. For example, a woman facing domestic violence can describe her situation and receive guidance on what legal protections are available — 24 hours a day, on her own phone.

2. **Document preparation**
   AI can help people fill out legal forms and applications (like protection orders or bail applications) by converting their own words into legally compliant language. A lawyer then reviews the output, but the heavy lifting is done.

3. **Connecting people with free lawyers**
   AI can quickly match someone's specific legal problem with available pro bono lawyers who have the right expertise.

4. **Making lawyers more efficient**
   When free legal help IS available, lawyers are often overwhelmed. AI tools can handle research and drafting, freeing lawyers to help more people.

**Important honest caveats:**
- AI should ASSIST, not replace, human lawyers for serious legal matters
- The quality of AI legal tools depends on the quality of the legal data they access
- Not everyone has a smartphone or internet access, so these tools must be designed with accessibility in mind
- AI must be tested carefully to avoid giving wrong advice, which could harm vulnerable people

The goal isn't to replace lawyers — it's to make justice accessible to the 90% of people who currently can't reach one."""
    },
]

print(f"Created {len(training_data)} training examples")

Created 6 training examples


In [23]:
# Format Data for the Trainer
from datasets import Dataset

def format_for_training(example):
    """Convert our data to the chat format the model expects"""
    message = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(message, tokenize=False)
    return {'text': text}

# Create Hugging Face Dataset
dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_for_training)

print("Dataset formatted for training:")
print(f"  Number of examples: {len(dataset)}")
print(f"  Columns: {dataset.column_names}")
print(f"\n  First example preview (first 200 chars):")
print(f"  {dataset[0]['text'][:200]}...")

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset formatted for training:
  Number of examples: 6
  Columns: ['instruction', 'response', 'text']

  First example preview (first 200 chars):
  <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
A woman is being abused by her partner and wants to know her options. Respond with em...


In [24]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantised model for training
model = prepare_model_for_kbit_training(model)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,                          # Rank: higher = more expressive but more memory
                                   # 8-32 is typical; 16 is a good default
    lora_alpha=32,                 # Scaling factor: usually 2x the rank
                                   # Controls how much LoRA adapters influence the output
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
                                   # Which layers to add LoRA to
                                   # Targeting attention + MLP layers for best results (7 pairs of A,B matrices total)
    lora_dropout=0.05,             # Dropout for regularisation
    bias="none",                   # Don't train bias terms
    task_type="CAUSAL_LM",         # We're doing causal language modelling
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Now check trainable parameters — should be much smaller!
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [31]:
# Quick re-initailize the model (for tunning hyperparameters)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
print("Model Re-initialize Completed")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model Re-initialize Completed
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [32]:
from trl import SFTTrainer, SFTConfig

# Training configuration
training_args = SFTConfig(
    output_dir="./legal-lora-model",        # Where to save checkpoints
    num_train_epochs=20,                    # Number of passes through the data
    per_device_train_batch_size=2,          # Batch size
    gradient_accumulation_steps=1,          # Accumulate gradients to simulate larger batch
    learning_rate=2e-4,                     # Learning rate: 1e-4 to 3e-4 is typical for LoRA
    warmup_steps=10,                        # Gradual LR warmup
    logging_steps=5,                        # Log every 5 steps
    save_strategy="epoch",                  # Save after each epoch
    save_total_limit=3,                     # Only keep last 3 checkpoints
    bf16=True,                              # <--- CHANGED: Use BFloat16 instead of FP16
    fp16=False,                             # <--- CHANGED: Ensure standard FP16 is disabled
    optim="adamw_torch_fused",              # <--- NEW: Fused optimizer runs 20% faster on Ampere
    dataset_text_field="text",              # Which field in dataset contains the text
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    # max_seq_length=1024,                    # Max sequence length (not supported param)
)
print("Starting training...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
5,2.162432
10,1.844938
15,1.205897
20,0.739627
25,0.368918
30,0.121502
35,0.030941
40,0.019502
45,0.013161
50,0.009987


TrainOutput(global_step=60, training_loss=0.5445955207571387, metrics={'train_runtime': 59.0934, 'train_samples_per_second': 2.031, 'train_steps_per_second': 1.015, 'total_flos': 347567649368064.0, 'train_loss': 0.5445955207571387})

In [ ]:
model.eval()  # ensure eval mode (CRITICAL)

# Override the ask_model function
def ask_model(prompt, model, tokenizer, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(model.device)

    # Set pad_token if missing
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.2,       # <--- penalise repetition
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

test_questions = [
    "A woman in Malawi is experiencing domestic violence. What legal steps can she take to protect herself? Respond in a structured, empathetic way.",
    "Explain what a protection order is and how to apply for one. Use simple language suitable for someone with no legal background.",
    "A journalist has been detained for reporting on government corruption. What are their rights under international law?",
]

In [39]:
# Check the response after LoRA
print("FINE-TUNED MODEL RESPONSES (AFTER LoRA)")

for i, q in enumerate(test_questions):
    print(f"\n--- Question {i+1} ---")
    print(f"Q: {q[:80]}...")
    response = ask_model(q, model, tokenizer)
    print(f"A: {response[:500]}")
    print()

# Also test with a NEW question the model hasn't seen
new_question = "A refugee family needs help understanding their legal rights in a new country. How would you explain the asylum process in simple terms?"

print("\n--- NEW Question (not in training data) ---")
print(f"Q: {new_question}")
response = ask_model(new_question, model, tokenizer)
print(f"A: {response[:500]}")

FINE-TUNED MODEL RESPONSES (AFTER LoRA)

--- Question 1 ---
Q: A woman in Malawi is experiencing domestic violence. What legal steps can she ta...
A: I'm sorry to hear about this situation. Your safety is the most important thing, and there are legal protections available.

**Immediate Steps:**
1. If you are in immediate danger, contact emergency services right away.
2. Reach out to your local women's shelter or domestic violence hotline for safe accommodation.

**Legal Options:**
A protection order (also called a restraining order) is a court-order that legally requires the abuser to stay away from you. To apply for one:
- Contact your neare


--- Question 2 ---
Q: Explain what a protection order is and how to apply for one. Use simple language...
A: A protection order (also called a restraining or injunction) is simply a court order — backed by power of law — that legally requires the person who hurt you to stay away from you.

**What kind of harm has happened?**
A protection order i

## Analysis of the Post-Trained Model Output
### Q1:
The tone completely transformed with empathy words at the beginning *"I'm sorry to hear about this situation"*, then provide **structured output** with clear headings.
> Correctly avoids fabricating specific phone number. The syle alignment working as intended, need RAG to retrieve the hotline numbers.

### Q2:
It gives a clear definition, explains when a protection order applies, and provides **concrete examples** of qualifying harm.
> No more "talk to a grown-up" tone. The model now speaks respectfully to adults. But it says "Also called a restraining or injunction", which is **not interchangeable** across jurisdictions, they have distinct legal meanings. Could solve that by filtering RAG with jurisdiction-specific documents.

### Q3:
**Structured** output. The framing "Reporting as a journalist is protected activity under International Law" is **strong and correct** in principle. However, this output **miss the citation** for Article 19 ICCPR comparing to the base model.
> This trade-off improve the format but loss the grounding material citation, demonstrating the need for SFT+RAG.

### Q4 (Unseen):
Have a structured output, showing effectivenness of learning the "style template" with LoRA SFT. HOWEVER, the model **switched to Chinese** mid-sentence *"continued safety while他们准备继续他们的生活"*.
> Indicating the multilingual weights leak through.

---

In [40]:
# Save the LoRA adapter (only the small trained weights)
model.save_pretrained("./legal-lora-adapter")
tokenizer.save_pretrained("./legal-lora-adapter")

import os
adapter_size = sum(
    os.path.getsize(os.path.join("./legal-lora-adapter", f))
    for f in os.listdir("./legal-lora-adapter")
    if os.path.isfile(os.path.join("./legal-lora-adapter", f))
)
print(f"LoRA adapter size: {adapter_size / 1e6:.1f} MB")

LoRA adapter size: 48.4 MB


## Takeaways
1. Fine-tuning successfully transferred the response style to unseen topics
2. Fine-tuning cannot provide reliable facts — "acceding" is wrong, the Chinese leak shows instability
3. A RAG system would have grounded the truth with citations in actual law documents, preventing word-interchanging and gauranteed citation.